# Data transformation

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

"""
Load the cleaned training and test datasets.
Concatenate them into a single DataFrame (df_all) to ensure consistent 
feature engineering and encoding across both sets.
"""
train_df = pd.read_csv('train_cleaned.csv')
test_df = pd.read_csv('test_cleaned.csv')

df_all = pd.concat([train_df, test_df], ignore_index=True)

In [2]:
"""
Create numerical features and scale them:
1. Apply log transformation (log1p) to 'AgeInMonths' to handle skewness.
2. Convert 'DateTime' to a continuous numerical feature ('DaysSinceStart') 
   representing the number of days since the earliest recorded date.
3. Standardize these new numerical features using StandardScaler.
"""
df_all['AgeInMonths_log'] = np.log1p(df_all['AgeInMonths'])

df_all['DateTime'] = pd.to_datetime(df_all['DateTime'])
min_date = df_all['DateTime'].min()
df_all['DaysSinceStart'] = (df_all['DateTime'] - min_date).dt.total_seconds() / (24 * 3600)

scaler = StandardScaler()
df_all[['AgeInMonths_scaled', 'DaysSinceStart_scaled']] = scaler.fit_transform(
    df_all[['AgeInMonths_log', 'DaysSinceStart']]
)

In [3]:
"""
Encode categorical features for machine learning models:
1. Apply One-Hot Encoding to low-cardinality features ('AnimalType', 'SexuponOutcome').
2. Apply Frequency Encoding to high-cardinality features ('Breed', 'Color') 
   to replace categories with their relative frequencies.
3. Create a binary feature ('has_name') indicating if the animal has a specific name.
"""
df_transformed = pd.get_dummies(df_all, columns=['AnimalType', 'SexuponOutcome'], drop_first=True)

for col in ['Breed', 'Color']:
    freq_map = df_all[col].value_counts(normalize=True).to_dict()
    df_transformed[f'{col}_freq'] = df_all[col].map(freq_map)

df_transformed['has_name'] = (df_transformed['Name'] != 'No Name').astype(int)

In [4]:
"""
Drop original and intermediate columns that are no longer needed.
Split the combined dataset back into the original training and test sets 
based on the presence of the target variable ('OutcomeType').
Finally, drop set-specific redundant columns and save the prepared datasets.
"""
cols_to_drop = ['Name', 'DateTime', 'AgeuponOutcome', 'AgeInMonths', 'AgeInMonths_log', 'DaysSinceStart', 'Breed', 'Color', 'AnimalID']
df_final = df_transformed.drop(columns=[c for c in cols_to_drop if c in df_transformed.columns])

train_transformed = df_final[df_final['OutcomeType'].notnull()].copy()
test_transformed = df_final[df_final['OutcomeType'].isnull()].copy()

train_transformed = train_transformed.drop(columns=['ID'], errors='ignore')
test_transformed = test_transformed.drop(columns=['OutcomeType', 'OutcomeSubtype', 'fold'], errors='ignore')

train_transformed.to_csv('train_transformed.csv', index=False)
test_transformed.to_csv('test_transformed.csv', index=False)

print(f"Transformed Train shape: {train_transformed.shape}")
print(f"Transformed Test shape: {test_transformed.shape}")

Transformed Train shape: (26729, 13)
Transformed Test shape: (11456, 11)
